Klasifikasi jenis buah (Apel / Jeruk / Mangga) berdasarkan berat (gram) dan kadar manis (skala 1–10).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder

In [ ]:
url = "https://raw.githubusercontent.com/plotly/datasets/master/diabetes.csv"
df_diabetes = pd.read_csv(url)

print("Bentuk dataset:", df_diabetes.shape)
print("\nLima data pertama:")
print(df_diabetes.head())
print("\nInfo dataset:")
print(df_diabetes.info())
print("\nDistribusi label Outcome:")
print(df_diabetes['Outcome'].value_counts())

In [ ]:
print(df_diabetes.describe().round(2))

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
features = df_diabetes.columns[:-1]
for i, col in enumerate(features):
    ax = axes[i//4][i%4]
    df_diabetes[df_diabetes['Outcome']==0][col].hist(ax=ax, alpha=0.6, label='Tidak Diabetes', bins=20, color='steelblue')
    df_diabetes[df_diabetes['Outcome']==1][col].hist(ax=ax, alpha=0.6, label='Diabetes', bins=20, color='tomato')
    ax.set_title(col)
    ax.legend(fontsize=7)
plt.suptitle('Distribusi Fitur per Kelas', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.pipeline import Pipeline

# Pisahkan fitur dan target
X_dia = df_diabetes.drop('Outcome', axis=1)
y_dia = df_diabetes['Outcome']

# Split data: 80% train, 20% test
X_train_d, X_test_d, y_train_d, y_test_d = train_test_split(
    X_dia, y_dia, test_size=0.2, random_state=42, stratify=y_dia
)

print(f"Ukuran data training : {X_train_d.shape}")
print(f"Ukuran data testing  : {X_test_d.shape}")

In [ ]:
k_values = range(1, 21)
train_acc = []
test_acc  = []

for k in k_values:
    scaler_d = StandardScaler()
    X_tr_s = scaler_d.fit_transform(X_train_d)
    X_te_s = scaler_d.transform(X_test_d)

    knn_k = KNeighborsClassifier(n_neighbors=k)
    knn_k.fit(X_tr_s, y_train_d)
    train_acc.append(accuracy_score(y_train_d, knn_k.predict(X_tr_s)))
    test_acc.append(accuracy_score(y_test_d, knn_k.predict(X_te_s)))

plt.figure(figsize=(9, 5))
plt.plot(k_values, train_acc, 'o-', label='Train Accuracy', color='steelblue')
plt.plot(k_values, test_acc, 's-', label='Test Accuracy', color='tomato')
best_k = k_values[np.argmax(test_acc)]
plt.axvline(best_k, linestyle='--', color='gray', alpha=0.6, label=f'Best k={best_k}')
plt.xlabel('Nilai K')
plt.ylabel('Akurasi')
plt.title('Perbandingan Akurasi KNN untuk Berbagai Nilai K')
plt.xticks(k_values)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nK terbaik: {best_k} dengan Test Accuracy: {max(test_acc):.4f}")

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

scaler_final = StandardScaler()
X_train_scaled = scaler_final.fit_transform(X_train_d)
X_test_scaled  = scaler_final.transform(X_test_d)

knn_final = KNeighborsClassifier(n_neighbors=best_k)
knn_final.fit(X_train_scaled, y_train_d)

y_pred_dia = knn_final.predict(X_test_scaled)

print(f"=== Hasil Model KNN Final (k={best_k}) ===")
print(f"Akurasi : {accuracy_score(y_test_d, y_pred_dia):.4f}")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test_d, y_pred_dia))
print("\nClassification Report:")
print(classification_report(y_test_d, y_pred_dia, target_names=['Tidak Diabetes','Diabetes']))

In [ ]:
import seaborn as sns

cm_dia = confusion_matrix(y_test_d, y_pred_dia)
plt.figure(figsize=(6, 5))
sns.heatmap(cm_dia, annot=True, fmt='d', cmap='YlOrRd',
            xticklabels=['Tidak Diabetes','Diabetes'],
            yticklabels=['Tidak Diabetes','Diabetes'])
plt.xlabel('Prediksi')
plt.ylabel('Aktual')
plt.title(f'Confusion Matrix KNN (k={best_k}) — Diabetes Dataset')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(9, 7))
corr = df_diabetes.corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Korelasi Antar Fitur — Diabetes Dataset')
plt.tight_layout()
plt.show()

In [ ]:
pasien_baru = pd.DataFrame([{
    'Pregnancies': 3,
    'Glucose': 148,
    'BloodPressure': 72,
    'SkinThickness': 35,
    'Insulin': 0,
    'BMI': 33.6,
    'DiabetesPedigreeFunction': 0.627,
    'Age': 50
}])

pasien_scaled = scaler_final.transform(pasien_baru)
pred_pasien = knn_final.predict(pasien_scaled)
prob_pasien = knn_final.predict_proba(pasien_scaled)[0]

status_pasien = 'DIABETES' if pred_pasien[0] == 1 else 'TIDAK DIABETES'
print(f"Hasil prediksi pasien baru : {status_pasien}")
print(f"  Probabilitas tidak diabetes : {prob_pasien[0]:.1%}")
print(f"  Probabilitas diabetes       : {prob_pasien[1]:.1%}")

**ANALISIA :**

**Dataset:**
- Dataset Pima Indians Diabetes terdiri dari 768 sampel dengan 8 fitur medis dan 1 label biner.
- Terdapat ketidakseimbangan kelas: ~65% tidak diabetes vs ~35% diabetes.

**Preprocessing:**
- Normalisasi dengan `StandardScaler` sangat diperlukan karena fitur memiliki satuan yang berbeda (misalnya Glucose dalam mg/dL vs usia dalam tahun).
- Parameter `stratify=y` digunakan saat split agar proporsi kelas terjaga di train dan test set.

**Pemilihan K:**
- Dilakukan percobaan K dari 1 sampai 20. K terlalu kecil (k=1) menghasilkan overfitting (train accuracy tinggi, test rendah).
- K optimal dipilih berdasarkan test accuracy tertinggi untuk menghindari overfitting.

**Interpretasi Confusion Matrix:**
- **True Positive (TP)**: Pasien diabetes diprediksi diabetes dengan benar → sangat penting di bidang medis.
- **False Negative (FN)**: Pasien diabetes diprediksi tidak diabetes → berbahaya karena pasien tidak mendapat penanganan.
- Dalam konteks medis, **recall kelas diabetes** lebih penting dari accuracy secara keseluruhan.

**Dari heatmap korelasi:**
- Glucose memiliki korelasi tertinggi dengan Outcome, menjadikannya fitur paling penting.
- BMI dan Age juga berkorelasi positif dengan diabetes.

**Kesimpulan Keseluruhan:**
- Regresi linear cocok untuk prediksi nilai kontinu (harga, omzet).
- Regresi logistik cocok untuk klasifikasi biner dengan interpretabilitas koefisien yang baik.
- KNN adalah algoritma non-parametrik yang fleksibel namun sensitif terhadap skala fitur dan pilihan K.
- Normalisasi fitur adalah langkah wajib sebelum menerapkan KNN.